# Transaction Risk Lab: reproduksi di Kaggle

Notebook ini menjalankan protokol yang sama dengan `txrisk run` di repo. **Belum pernah dijalankan** saat repo dibuat: Kaggle tidak bisa dijangkau dari lingkungan pengembangan. Hasil Kaggle hanya boleh dikutip setelah notebook ini benar-benar dijalankan dan keluarannya disimpan.

Dua bagian:

1. **Reproduksi utama.** Membangkitkan data Sparkov yang sama (seed tetap), lalu melatih dan mengevaluasi. Angka harus sama dengan `reports/results.json` di repo dalam toleransi 0,005.
2. **Replikasi pada dataset Kaggle publik** `kartik2112/fraud-detection` (CC0), keluaran Sparkov versi lama dengan 1.000 nasabah pada tanggal yang sama. Ini sampel independen dari generator yang sama, bukan data nyata.

Pengaturan: Internet **On** (untuk `pip install` dari GitHub). GPU tidak diperlukan.

In [ ]:
REPO = "https://github.com/muhshandy-dev/transaction-risk-lab"
!pip install -q "git+{REPO}"
!git clone -q --depth 1 {REPO} /kaggle/working/repo

## 1. Reproduksi utama

In [ ]:
!cd /kaggle/working && txrisk run --data /kaggle/working/data --out /kaggle/working/run/reports --threads 4

In [ ]:
!python /kaggle/working/repo/tools/compare_reports.py /kaggle/working/repo/reports /kaggle/working/run/reports

In [ ]:
import json
r = json.load(open('/kaggle/working/run/reports/results.json'))
for k, m in r['models'].items():
    print(f"{m['label']:50s} PR-AUC {m['ap']:.3f}  recall@1% {m['at_budget']['0.01']['recall']:.3f}")
print(json.dumps(r['paired_lgbm_minus_logistic']['ap'], indent=1))

## 2. Replikasi pada dataset Kaggle `kartik2112/fraud-detection`

Tambahkan dataset itu lewat **Add Input**. Berkas yang dipakai: `fraudTrain.csv` dan `fraudTest.csv`. Protokol, pembagian waktu, dan grid hiperparameter tidak diubah.

In [ ]:
import glob
files = sorted(glob.glob('/kaggle/input/**/fraud*.csv', recursive=True))
print(files)
assert len(files) == 2, 'tambahkan dataset kartik2112/fraud-detection sebagai input'
!txrisk import-kaggle {files[0]} {files[1]} --data /kaggle/working/kaggle_data
!txrisk run --data /kaggle/working/kaggle_data --out /kaggle/working/kaggle_run/reports --threads 4

In [ ]:
r2 = json.load(open('/kaggle/working/kaggle_run/reports/results.json'))
for k, m in r2['models'].items():
    print(f"{m['label']:50s} PR-AUC {m['ap']:.3f}  recall@1% {m['at_budget']['0.01']['recall']:.3f}")
print(json.dumps(r2['paired_lgbm_minus_logistic'], indent=1))

Simpan folder `/kaggle/working/run` dan `/kaggle/working/kaggle_run` sebagai Output notebook, lalu catat versi notebook di README sebelum mengutip angka apa pun.